#P1 · Calidad y Limpieza de datos
## Dataset Airbnb Buenos Aires - Listings.csv.gz
**Fuente:** Inside Airbnb
**Fecha de Scrape:** Enero 2026
**Objetivo:** Identificar y corregir problemas de calidad para dejar el dataset listo para análisis

## Diagnóstico de calidad de datos

### Dimensiones

- Filas: 27.348
- Columnas: 85
- Duplicados: 0

### Problemas encontrados

1. **15 columnas con 100% de nulos** - eliminar
2. **Columnas 'license' (98% nulos) y 'host_about' (43% nulos)** - eliminar
3. **Columna 'review_scores_*' (12% nulos)** - corresponden a propiedades sin reseña, imputar con 0 o dejar como NaN según el análisis
4. **Columnas 'beds', 'bedrooms' y 'bathrooms' (<1% nulos)** - imputar con mediana

### Próximos pasos

- Eliminar columnas con >40% de nulos
- Imputar columnas críticas
- Convertir columnas de fecha a tipo datetime

**Limitación importante:** la columna "price" está 100% vacía en este scrape.
El dataset no permite análisis de precios.

### Insight destacado
847 propiedades (3.52%) llevan más de 3 años sin reseñas. Son listings activos, pero sin actividad real.


In [ ]:
import pandas as pd
import sys
sys.path.append("..")

from src.clean import imputar_por_grupo,parsear_fechas,crear_variables_fecha


df = pd.read_csv("../data/raw/listings.csv.gz")

print (df.dtypes)
print (df.shape)

id                                                int64
listing_url                                      object
scrape_id                                         int64
last_scraped                                     object
source                                           object
                                                 ...   
calculated_host_listings_count                    int64
calculated_host_listings_count_entire_homes       int64
calculated_host_listings_count_private_rooms      int64
calculated_host_listings_count_shared_rooms       int64
reviews_per_month                               float64
Length: 85, dtype: object
(27348, 85)


In [2]:
print(df["last_scraped"].unique())

['2026-01-25' '2026-01-26']


In [3]:
# Calcula el total de valores nulos por variable

df.isnull().sum().sort_values(ascending=False).head(20)

neighborhood_overview           27348
host_since                      27348
host_acceptance_rate            27348
host_response_rate              27348
host_response_time              27348
host_total_listings_count       27348
host_verifications              27348
neighbourhood_group_cleansed    27348
instant_bookable                27348
calendar_updated                27348
estimated_revenue_l365d         27348
price                           27348
neighbourhood                   27348
host_thumbnail_url              27348
host_neighbourhood              27348
license                         26887
host_about                      11784
host_location                    5783
review_scores_accuracy           3304
reviews_per_month                3303
dtype: int64

In [4]:
# Calcula el porcentaje de nulos por columna
# y muestra solo las que tienen al menos 1 nulo

nulos_pct = (df.isnull().sum()/len(df))*100
nulos_pct = nulos_pct.round(2)
nulos_pct = nulos_pct[nulos_pct > 0]
nulos_pct = nulos_pct.sort_values(ascending=False)
print(nulos_pct)

neighborhood_overview           100.00
host_since                      100.00
host_response_rate              100.00
host_response_time              100.00
host_acceptance_rate            100.00
host_total_listings_count       100.00
host_neighbourhood              100.00
host_thumbnail_url              100.00
estimated_revenue_l365d         100.00
instant_bookable                100.00
price                           100.00
host_verifications              100.00
neighbourhood                   100.00
neighbourhood_group_cleansed    100.00
calendar_updated                100.00
license                          98.31
host_about                       43.09
host_location                    21.15
review_scores_value              12.08
review_scores_rating             12.08
first_review                     12.08
last_review                      12.08
review_scores_accuracy           12.08
review_scores_location           12.08
reviews_per_month                12.08
review_scores_communicati

In [5]:
# Cuántas filas duplicadas hay en el dataset

print(df.duplicated().sum())

# Ver las primeras 5 filas de las columnas más importantes

cols = ["id","name","room_type","beds","bedrooms","bathrooms"]
print(df[cols].head())

0
                    id                                               name  \
0             42610838  Puerto Madero a 3 cuadras, centro, bello , tea...   
1  1305876403852901802                       Apart estudio en Microcentro   
2  1542233033640525302      Departamento en Buenos Aires, abasto shopping   
3  1004530078359434134                           Departamento en Recoleta   
4   800145927121871422                            Coqueto para 4 personas   

         room_type  beds  bedrooms  bathrooms  
0  Entire home/apt   1.0       1.0        1.0  
1  Entire home/apt   1.0       1.0        1.0  
2  Entire home/apt   2.0       1.0        1.0  
3  Entire home/apt   1.0       1.0        1.0  
4  Entire home/apt   3.0       2.0        1.0  


In [6]:
# Columnas con más del 40% de nulos no aportan información útil
# Calcular el umbral y eliminar

umbral = 0.40
cols_a_eliminar = df.columns[df.isnull().mean()>umbral].tolist()

print(f"Columnas a eliminar: {len(cols_a_eliminar)}")
print(cols_a_eliminar)

Columnas a eliminar: 17
['neighborhood_overview', 'host_since', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_group_cleansed', 'price', 'calendar_updated', 'estimated_revenue_l365d', 'license', 'instant_bookable']


In [7]:
# Es raro que la columna price tenga tantos valores nulos. Se explora la variable

print(df["price"].head(20))


0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
5    NaN
6    NaN
7    NaN
8    NaN
9    NaN
10   NaN
11   NaN
12   NaN
13   NaN
14   NaN
15   NaN
16   NaN
17   NaN
18   NaN
19   NaN
Name: price, dtype: float64


In [8]:
# Probablemente airbnb no incluya el precio. Se busca si hay variables que contengan la variable price en el nombre

cols_precio = [cols for cols in df.columns if "price" in cols.lower()]
print(cols_precio)



['price']


In [9]:
# Se elimina las columnas con más del 40% de nulos
# Inplace = False significa que no modifica el df directamente
# sino que devuelve un df nuevo llamado "data_clean"

df_clean = df.drop(columns=cols_a_eliminar)

print(f"Dimensiones originales: {df.shape}")
print (f"Dimensiones después: {df_clean.shape}")

Dimensiones originales: (27348, 85)
Dimensiones después: (27348, 68)


In [10]:
# Verificar los nulos que quedan en df df_clean

nulos_restantes = (df_clean.isnull().sum()/len(df_clean))*100
nulos_restantes = nulos_restantes.round(2)
nulos_restantes = nulos_restantes[nulos_restantes>0]
nulos_restantes = nulos_restantes.sort_values(ascending=False)

print(nulos_restantes)

host_location                  21.15
review_scores_cleanliness      12.08
review_scores_checkin          12.08
review_scores_communication    12.08
review_scores_location         12.08
last_review                    12.08
first_review                   12.08
review_scores_value            12.08
reviews_per_month              12.08
review_scores_accuracy         12.08
review_scores_rating           12.08
description                     2.18
has_availability                0.75
minimum_nights                  0.39
maximum_maximum_nights          0.39
maximum_nights                  0.39
minimum_minimum_nights          0.39
maximum_minimum_nights          0.39
minimum_maximum_nights          0.39
bedrooms                        0.22
bathrooms_text                  0.04
beds                            0.03
bathrooms                       0.03
dtype: float64


In [11]:
# Tenemos 3 grupos: variables >15%, 15%< y >3% y variables 2%<
# Del primer grupo le imputamos valor binario, de la segunda podemos prescindir o imputarle NaN y de la tercera imputación por grupos
# Ver qué valores tiene la variable room_type para imputar por grupos

print(df_clean["room_type"].value_counts())

room_type
Entire home/apt    24890
Private room        2249
Shared room          132
Hotel room            77
Name: count, dtype: int64


In [12]:
# Mediana de beds, bedrooms y bathrooms agrupado por room_type
cols_imputar = ["beds", "bedrooms", "bathrooms"]

print(df_clean.groupby("room_type")[cols_imputar].median())

                 beds  bedrooms  bathrooms
room_type                                 
Entire home/apt   2.0       1.0        1.0
Hotel room        0.0       1.0        1.0
Private room      1.0       1.0        1.0
Shared room       2.0       1.0        1.0


In [13]:
# Es raro que el valor hotel room tenga mediana = 0 en beds
# Ver cuántos hotel room tienen beds valor 0 vs valores reales

print(df_clean[df_clean["room_type"] == "Hotel room"]["beds"].value_counts(dropna=False))

beds
0.0     50
2.0     10
1.0      6
3.0      4
6.0      2
5.0      2
9.0      1
4.0      1
16.0     1
Name: count, dtype: int64


In [14]:
# Imputamos beds, bedrooms y bathrooms por mediana de room_type
# pero solo para los tipos que no son Hotel Room

tipos_a_imputar = ["Entire home/apt","Private room","Shared room"]

for col in ["beds","bedrooms","bathrooms"]:
    for tipo in tipos_a_imputar:
        mediana = df_clean.loc[df_clean["room_type"] == tipo, col].median()
        mask = (df_clean["room_type"] == tipo) & (df_clean[col].isnull())
        df_clean.loc[mask,col] = mediana

print("Nulos restantes en beds, bedrooms y bathrooms:")
print(df_clean[["beds","bedrooms","bathrooms"]].isnull().sum())


Nulos restantes en beds, bedrooms y bathrooms:
beds         0
bedrooms     0
bathrooms    0
dtype: int64


In [15]:
# Ahora, convertir columnas de fecha a tipo datetime
# Ver qué columnas de fecha tenemos

cols_fecha = ["last_scraped","first_review","last_review"]

print(df_clean[cols_fecha].dtypes)
print(df_clean[cols_fecha].head())

last_scraped    object
first_review    object
last_review     object
dtype: object
  last_scraped first_review last_review
0   2026-01-25          NaN         NaN
1   2026-01-25   2025-01-18  2025-01-18
2   2026-01-25          NaN         NaN
3   2026-01-25   2023-10-29  2025-12-07
4   2026-01-25          NaN         NaN


In [16]:
for col in cols_fecha:
    df_clean[col] = pd.to_datetime(df_clean[col])

print(df_clean[col].dtypes)

datetime64[ns]


In [17]:
# Creamos una variable binaria que indica si la propiedad tiene reseña o no
# notna() devuelve TRUE si el valor no es nulo, FALSE si el valor es nulo
# astype(int) convierte TRUE -> 1 y FALSE -> 0

df_clean["tiene_resenas"] = df_clean["review_scores_rating"].notna().astype(int)

print(df_clean["tiene_resenas"].value_counts())

tiene_resenas
1    24045
0     3303
Name: count, dtype: int64


In [18]:
# En relación con las otras variables relacionadas con review podemos derivar nuevas variables:
# first_review + last_review → dias_activo (rango temporal)
# last_review + last_scraped → dias_desde_ultima_resena (qué tan reciente es la actividad)
# first_review + last_scraped → antiguedad_total (cuánto lleva en la plataforma)

print(df_clean[["first_review","last_review","last_scraped"]].describe())

                        first_review                    last_review  \
count                          24045                          24045   
mean   2022-12-19 07:00:31.890205952  2025-08-01 16:06:02.919525888   
min              2010-05-19 00:00:00            2013-01-06 00:00:00   
25%              2022-05-16 00:00:00            2025-10-09 00:00:00   
50%              2023-10-21 00:00:00            2025-12-20 00:00:00   
75%              2024-12-02 00:00:00            2026-01-08 00:00:00   
max              2026-01-25 00:00:00            2026-01-25 00:00:00   

                        last_scraped  
count                          27348  
mean   2026-01-25 11:36:53.075910656  
min              2026-01-25 00:00:00  
25%              2026-01-25 00:00:00  
50%              2026-01-25 00:00:00  
75%              2026-01-26 00:00:00  
max              2026-01-26 00:00:00  


In [19]:
# dias_activo: rango entre primera y última celda
# solo tiene sentido para propiedades con reseñas

df_clean["dias_activo"] = (
    df_clean["last_review"] - df_clean["first_review"]
).dt.days

#días_desde_ultima_resena: qué tan reciente es la actividad
df_clean["dias_desde_ultima_resena"] = (
    df_clean["last_scraped"] - df_clean["last_review"]
).dt.days

#antiguedad_total: cuánto lleva en la plataforma desde primera reseña
df_clean["antiguedad_total"] = (
    df_clean["last_scraped"] - df_clean["first_review"]
).dt.days

print(df_clean[["dias_activo","dias_desde_ultima_resena","antiguedad_total"]].describe())

        dias_activo  dias_desde_ultima_resena  antiguedad_total
count  24045.000000              24045.000000      24045.000000
mean     956.378831                176.808318       1133.187149
std     1001.766464                406.624982       1047.233309
min        0.000000                  0.000000          0.000000
25%      251.000000                 17.000000        419.000000
50%      678.000000                 36.000000        827.000000
75%     1182.000000                109.000000       1351.000000
max     5703.000000               4768.000000       5731.000000


In [20]:
# Cuántas propiedades llevan más de 3 años sin recibir una reseña

sin_actividad = (df_clean["dias_desde_ultima_resena"] > 365 * 3).sum()
total_con_resenas = df_clean["tiene_resenas"].sum()

print(f"Propiedades sin reseña hace más de 3 años: {sin_actividad}")
print(f"Como % del total con reseñas: {round(sin_actividad/total_con_resenas*100,2)}%")

Propiedades sin reseña hace más de 3 años: 847
Como % del total con reseñas: 3.52%


In [21]:
# Guardamos el dataset limpio en data/processed/

df_clean.to_csv("../data/processed/listings_clean.csv", index=False)

print(f"Dataset guardado: {df_clean.shape[0]:,} filas x {df_clean.shape[1]} columnas")

Dataset guardado: 27,348 filas x 72 columnas


In [22]:
# Creamos un resumen comparativo antes y después de la limpieza
reporte = {
    "filas": {"antes": df.shape[0], "después": df_clean.shape[0]},
    "columnas": {"antes": df.shape[1], "después": df_clean.shape[1]},
    "duplicados": {"antes":df.duplicated().sum(), "después": df_clean.duplicated().sum()},
    "nulos_totales": {"antes":df.isnull().sum().sum(), "después": df_clean.isnull().sum().sum()},
                      "variables_nuevas": ["tiene_resenas", "dias_activo", "dias_desde_ultima_resena", "antiguedad_total"]
        }

for key,val in reporte.items():
    if key == "variables_nuevas":
        print(f"{key}: {val}")
    else:
        print(f"{key}: {val['antes']} -> {val['después']}")

filas: 27348 -> 27348
columnas: 85 -> 72
duplicados: 0 -> 0
nulos_totales: 489229 -> 50171
variables_nuevas: ['tiene_resenas', 'dias_activo', 'dias_desde_ultima_resena', 'antiguedad_total']


In [29]:
# Guardamos el reporte en reports/quality_report.md
reporte_md = f"""# Reporte de calidad de datos
## Dataset: Airbnb Buenos Aires — enero 2026

## Resumen antes/después

| Métrica | Antes | Después |
|---|---|---|
| Filas | {df.shape[0]:,} | {df_clean.shape[0]:,} |
| Columnas | {df.shape[1]} | {df_clean.shape[1]} |
| Duplicados | {df.duplicated().sum()} | {df_clean.duplicated().sum()} |
| Nulos totales | {df.isnull().sum().sum():,} | {df_clean.isnull().sum().sum():,} |
| Reducción de nulos | | 89.7% |

## Variables nuevas creadas
- `tiene_resenas` — binaria (1/0)
- `dias_activo` — días entre primera y última reseña
- `dias_desde_ultima_resena` — días entre última reseña y scrape
- `antiguedad_total` — días desde primera reseña al scrape

## Decisiones de limpieza
1. Eliminadas 17 columnas con más del 40% de nulos
2. Imputados `beds`, `bedrooms`, `bathrooms` con mediana por `room_type`
3. `review_scores_*` conservados como NaN (propiedades sin reseñas)
4. Fechas convertidas a datetime

## Insight destacado
847 propiedades (3.52%) llevan más de 3 años sin reseñas.
"""

with open("../reports/quality_report.md", "w", encoding="utf-8") as f:
    f.write(reporte_md)

print("Reporte guardado en reports/quality_report.md")


Reporte guardado en reports/quality_report.md


In [30]:
# Verificamos que el archivo se creó correctamente

with open("../reports/quality_report.md", "r", encoding="utf-8") as f:
    print(f.read())

# Reporte de calidad de datos
## Dataset: Airbnb Buenos Aires — enero 2026

## Resumen antes/después

| Métrica | Antes | Después |
|---|---|---|
| Filas | 27,348 | 27,348 |
| Columnas | 85 | 72 |
| Duplicados | 0 | 0 |
| Nulos totales | 489,229 | 50,171 |
| Reducción de nulos | | 89.7% |

## Variables nuevas creadas
- `tiene_resenas` — binaria (1/0)
- `dias_activo` — días entre primera y última reseña
- `dias_desde_ultima_resena` — días entre última reseña y scrape
- `antiguedad_total` — días desde primera reseña al scrape

## Decisiones de limpieza
1. Eliminadas 17 columnas con más del 40% de nulos
2. Imputados `beds`, `bedrooms`, `bathrooms` con mediana por `room_type`
3. `review_scores_*` conservados como NaN (propiedades sin reseñas)
4. Fechas convertidas a datetime

## Insight destacado
847 propiedades (3.52%) llevan más de 3 años sin reseñas.



In [28]:
import duckdb

# Conectamos DuckDB directamente al CSV limpio
# DuckDB puede leer CSVs como si fueran tablas SQL

conn = duckdb.connect()

# Registramos el dataframe como tabla virtual

conn.register("listings", df_clean)

# Primera consulta: ¿cuántos listings hay por tipo de habitación?

resultado = conn.execute ("""
    SELECT
        room_type,
        COUNT(*) as total,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as porcentaje
    FROM listings
    GROUP BY room_type
    ORDER BY total DESC
""").df()

print(resultado)

         room_type  total  porcentaje
0  Entire home/apt  24890       91.01
1     Private room   2249        8.22
2      Shared room    132        0.48
3       Hotel room     77        0.28


In [33]:
# Segunda consulta: ¿Cuál es la mediana de dias_activo por room_type, solo para propiedades que tienen reseñas?

resultado_2 = conn.execute ("""
    SELECT
        room_type,
        MEDIAN(dias_activo) as mediana_dias_activo
    from listings
    WHERE 
        tiene_resenas = 1
    GROUP BY room_type
""").df()

# Tercera consulta: ¿Cuántas propiedades tienen más de 365 días sin reseña, agrupado por room_type?
resultado_3 = conn.execute ("""
    SELECT
        room_type,
        COUNT(*) as total
    from listings
    WHERE 
        dias_desde_ultima_resena > 365
    GROUP BY room_type
""").df()

print(resultado_2)
print(resultado_3)




         room_type  mediana_dias_activo
0       Hotel room               2224.0
1      Shared room                306.0
2  Entire home/apt                677.0
3     Private room                700.0
         room_type  total
0      Shared room     11
1       Hotel room      8
2  Entire home/apt   2708
3     Private room    395
